# Python OOP & Design Patterns for Data Engineering

## Purpose of This Notebook

This notebook teaches **Object-Oriented Programming** patterns that you'll use to build:
- **Maintainable ETL pipelines**
- **Reusable data processors**
- **Testable code with dependency injection**

| Pattern | Real-World Use Case |
|---------|--------------------|
| Encapsulation | Hide database connection details |
| Abstract Classes | Define ETL interface (Extract, Transform, Load) |
| Factory | Create right processor for file type (CSV, JSON, Parquet) |
| Singleton | Database connection pool (only one instance) |
| Strategy | Swap hashing algorithms without changing code |
| Decorator | Add logging/timing to any function |

---

In [ ]:
# IMPORTS
from abc import ABC, abstractmethod  # For abstract base classes
from dataclasses import dataclass, field  # Modern way to create data classes
from typing import List, Optional, Dict, Any  # Type hints
from datetime import datetime
from enum import Enum  # Type-safe constants
import functools  # For decorator utilities
import time

---
## 1. Encapsulation - Hiding Internal State

### What is Encapsulation?
- **Bundle data and methods** together in a class
- **Hide internal details** from outside code
- **Control access** via getter/setter methods

### Why Use It?
- Prevent bugs from direct data modification
- Validate data before allowing changes
- Change internal implementation without breaking external code

In [ ]:
class BankAccount:
    """
    ENCAPSULATION EXAMPLE:
    - _balance is PRIVATE (convention: underscore prefix)
    - External code can't directly set balance to negative!
    - All changes go through deposit() and withdraw() methods
    """
    
    def __init__(self, account_id: str, initial_balance: float = 0):
        self.account_id = account_id  # PUBLIC: can be read/written directly
        self._balance = initial_balance  # PRIVATE: underscore = "don't touch directly"
    
    @property
    def balance(self) -> float:
        """
        @property makes this a GETTER method.
        External code can do: account.balance (reads like an attribute)
        But internally it's a method call!
        """
        return self._balance
    
    def deposit(self, amount: float) -> None:
        """
        CONTROLLED ACCESS: We validate before modifying.
        This is ENCAPSULATION - we control how data changes.
        """
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        
        self._balance += amount  # Only valid deposits modify balance
        print(f"✅ Deposited ${amount}. New balance: ${self._balance}")
    
    def withdraw(self, amount: float) -> bool:
        """
        Returns True if successful, False if insufficient funds.
        Prevents negative balance!
        """
        if amount > self._balance:
            print(f"❌ Cannot withdraw ${amount}. Only ${self._balance} available.")
            return False
        
        self._balance -= amount
        print(f"✅ Withdrew ${amount}. New balance: ${self._balance}")
        return True

In [ ]:
# DEMO: Encapsulation in action
account = BankAccount("ACC001", 1000)

print(f"Account ID: {account.account_id}")
print(f"Balance: ${account.balance}")  # Uses @property getter

# Valid operations
account.deposit(500)
account.withdraw(200)

# Invalid operations are BLOCKED
try:
    account.deposit(-100)  # Will raise ValueError
except ValueError as e:
    print(f"❌ Error: {e}")

account.withdraw(5000)  # Will fail (insufficient funds)

# IMPORTANT: You CAN still access _balance directly...
# But the underscore convention says "DON'T DO THIS!"
print(f"\n⚠️ Direct access (bad practice): {account._balance}")

---
## 2. Abstract Base Classes - Defining Contracts

### What is an Abstract Class?
- A **blueprint** that defines what methods a class MUST have
- **Cannot be instantiated** directly
- Forces all subclasses to implement required methods

### Real-World Use: ETL Pipeline Interface
All data processors must have: `extract()`, `transform()`, `load()`

In [ ]:
class DataProcessor(ABC):
    """
    ABSTRACT BASE CLASS (ABC):
    - Defines the INTERFACE that all processors must follow
    - You CANNOT create a DataProcessor directly
    - You MUST create a subclass that implements all @abstractmethod
    
    This is the "Template Method" pattern:
    - run() defines the OVERALL workflow
    - Subclasses provide the SPECIFIC implementation
    """
    
    @abstractmethod  # This FORCES subclasses to implement this method
    def extract(self, source: str) -> Any:
        """Extract data from source. Subclass MUST implement."""
        pass
    
    @abstractmethod
    def transform(self, data: Any) -> Any:
        """Transform the data. Subclass MUST implement."""
        pass
    
    @abstractmethod
    def load(self, data: Any, destination: str) -> None:
        """Load data to destination. Subclass MUST implement."""
        pass
    
    def run(self, source: str, destination: str) -> None:
        """
        TEMPLATE METHOD: Defines the ETL workflow.
        This is NOT abstract - it's implemented here.
        All subclasses get this method for FREE.
        """
        print(f"\n{'='*50}")
        print(f"Running ETL: {source} → {destination}")
        print(f"{'='*50}")
        
        # Step 1: Extract
        print("📥 EXTRACT phase...")
        data = self.extract(source)
        
        # Step 2: Transform
        print("⚙️ TRANSFORM phase...")
        transformed = self.transform(data)
        
        # Step 3: Load
        print("📤 LOAD phase...")
        self.load(transformed, destination)

In [ ]:
class CSVProcessor(DataProcessor):
    """
    CONCRETE implementation for CSV files.
    Inherits from DataProcessor and implements ALL abstract methods.
    """
    
    def extract(self, source: str) -> List[Dict]:
        # In real code: pd.read_csv(source) or open(source)
        print(f"   Reading CSV file: {source}")
        return [{"id": 1, "name": "alice"}, {"id": 2, "name": "bob"}]
    
    def transform(self, data: List[Dict]) -> List[Dict]:
        # Transform: uppercase all names
        print(f"   Transforming {len(data)} rows...")
        for row in data:
            row["name"] = row["name"].upper()
        return data
    
    def load(self, data: List[Dict], destination: str) -> None:
        print(f"   Writing {len(data)} rows to: {destination}")
        for row in data:
            print(f"      {row}")


class JSONProcessor(DataProcessor):
    """
    CONCRETE implementation for JSON files.
    Different implementation, SAME interface.
    """
    
    def extract(self, source: str) -> Dict:
        print(f"   Parsing JSON file: {source}")
        return {"users": [{"id": 1}], "count": 1}
    
    def transform(self, data: Dict) -> Dict:
        print(f"   Adding timestamp to JSON...")
        data["processed_at"] = datetime.now().isoformat()
        return data
    
    def load(self, data: Dict, destination: str) -> None:
        print(f"   Writing JSON to: {destination}")
        import json
        print(f"      {json.dumps(data, indent=2)}")

In [ ]:
# DEMO: Polymorphism - Same interface, different behavior

def process_data(processor: DataProcessor, source: str, dest: str):
    """
    POLYMORPHISM: This function works with ANY DataProcessor.
    It doesn't care if it's CSV, JSON, Parquet, etc.
    """
    processor.run(source, dest)

# Same function, different processors
process_data(CSVProcessor(), "users.csv", "output.csv")
process_data(JSONProcessor(), "data.json", "output.json")

# TRY THIS: Creating DataProcessor directly will FAIL
try:
    abstract_processor = DataProcessor()  # Error!
except TypeError as e:
    print(f"\n❌ Cannot instantiate abstract class: {e}")

---
## 3. Dataclasses - Modern Data Containers

### What are Dataclasses?
- Python 3.7+ feature for classes that mainly store data
- Auto-generates: `__init__`, `__repr__`, `__eq__`
- Less boilerplate than regular classes

In [ ]:
# BEFORE: Regular class (lots of boilerplate)
class ClaimOld:
    def __init__(self, claim_id, policyholder_id, amount):
        self.claim_id = claim_id
        self.policyholder_id = policyholder_id
        self.amount = amount
    
    def __repr__(self):
        return f"ClaimOld({self.claim_id}, {self.policyholder_id}, {self.amount})"
    
    def __eq__(self, other):
        return (self.claim_id == other.claim_id and 
                self.policyholder_id == other.policyholder_id and
                self.amount == other.amount)

print("Old way (15 lines of code):")
old_claim = ClaimOld("CL001", "P1", 5000)
print(old_claim)

In [ ]:
# AFTER: Dataclass (much cleaner!)
@dataclass
class Claim:
    """
    @dataclass AUTOMATICALLY generates:
    - __init__(self, claim_id, policyholder_id, amount, claim_date, status)
    - __repr__ for nice printing
    - __eq__ for comparison
    """
    claim_id: str
    policyholder_id: str
    amount: float
    # Default values for optional fields
    claim_date: datetime = field(default_factory=datetime.now)
    status: str = "PENDING"
    
    def __post_init__(self):
        """
        __post_init__ runs AFTER the auto-generated __init__.
        Use it for VALIDATION!
        """
        if self.amount <= 0:
            raise ValueError(f"Amount must be positive, got {self.amount}")

print("\nNew way with @dataclass (5 lines of code):")
claim = Claim("CL001", "P1", 5000)
print(claim)
print(f"Status: {claim.status}")
print(f"Date: {claim.claim_date}")

In [ ]:
# IMMUTABLE dataclass with frozen=True
@dataclass(frozen=True)  # Makes it immutable (cannot change after creation)
class DatabaseConfig:
    """
    frozen=True means:
    - All attributes are READ-ONLY after creation
    - Can be used as dictionary keys
    - Perfect for configuration objects
    """
    host: str
    port: int = 5432
    database: str = "claims"

config = DatabaseConfig("localhost")
print(f"\nImmutable config: {config}")

try:
    config.host = "new-host"  # This will FAIL
except Exception as e:
    print(f"❌ Cannot modify frozen dataclass: {type(e).__name__}")

---
## 4. Factory Pattern - Creating Objects

### What is Factory Pattern?
- A class that creates OTHER objects
- Hides complex creation logic
- Decides WHICH class to instantiate at runtime

In [ ]:
class ProcessorFactory:
    """
    FACTORY PATTERN:
    - Centralizes object creation
    - Client code just says "I need a processor for CSV"
    - Factory figures out WHICH class to create
    
    Benefits:
    - Add new processor types without changing client code
    - Hide complex initialization logic
    """
    
    # Registry of available processors
    _processors = {
        "csv": CSVProcessor,
        "json": JSONProcessor,
    }
    
    @classmethod
    def register(cls, format_type: str, processor_class):
        """
        EXTENSIBILITY: Register new processor types at runtime!
        No need to modify this class to add new processors.
        """
        print(f"📦 Registered new processor: {format_type}")
        cls._processors[format_type] = processor_class
    
    @classmethod
    def create(cls, format_type: str) -> DataProcessor:
        """
        Create a processor based on format type.
        
        The client doesn't need to know about CSVProcessor, JSONProcessor, etc.
        They just ask for "csv" or "json" and get the right object.
        """
        processor_class = cls._processors.get(format_type.lower())
        
        if not processor_class:
            available = list(cls._processors.keys())
            raise ValueError(f"Unknown format: {format_type}. Available: {available}")
        
        print(f"🏭 Factory creating: {processor_class.__name__}")
        return processor_class()  # Create and return instance

In [ ]:
# DEMO: Using the Factory

# Client code doesn't import CSVProcessor or JSONProcessor!
# It just asks the factory for what it needs.

file_type = "csv"  # This could come from user input, config file, etc.
processor = ProcessorFactory.create(file_type)
processor.run("input.csv", "output.csv")

file_type = "json"
processor = ProcessorFactory.create(file_type)
processor.run("data.json", "results.json")

# Error handling
try:
    processor = ProcessorFactory.create("parquet")  # Not registered!
except ValueError as e:
    print(f"\n❌ {e}")

---
## 5. Singleton Pattern - One Instance Only

### What is Singleton?
- Ensures only **ONE instance** of a class exists
- All code shares the same instance

### Use Cases:
- Database connection pools
- Configuration managers  
- Logging systems

In [ ]:
class DatabaseConnection:
    """
    SINGLETON PATTERN:
    - Only ONE database connection exists
    - All code uses the SAME connection
    
    HOW IT WORKS:
    1. __new__ is called BEFORE __init__
    2. If instance doesn't exist, create it
    3. If instance exists, return the existing one
    """
    
    _instance = None  # Class variable (shared across all instances)
    _initialized = False
    
    def __new__(cls, *args, **kwargs):
        """
        __new__ creates the instance (called before __init__).
        We override it to return existing instance if it exists.
        """
        if cls._instance is None:
            # First time: create the instance
            print("🔧 Creating new database connection...")
            cls._instance = super().__new__(cls)
        else:
            print("♻️ Reusing existing connection...")
        return cls._instance
    
    def __init__(self, connection_string: str = "default"):
        """
        Only initialize ONCE (first time).
        Subsequent calls skip initialization.
        """
        if not DatabaseConnection._initialized:
            self.connection_string = connection_string
            self.connected = False
            DatabaseConnection._initialized = True
            print(f"   Connection string: {connection_string}")
    
    def connect(self):
        self.connected = True
        print("✅ Connected to database")
    
    def disconnect(self):
        self.connected = False
        print("🔌 Disconnected from database")

In [ ]:
# DEMO: Singleton in action

# Reset for demo
DatabaseConnection._instance = None
DatabaseConnection._initialized = False

# First call: creates new instance
db1 = DatabaseConnection("postgres://localhost:5432/claims")

# Second call: returns SAME instance (connection string is IGNORED!)
db2 = DatabaseConnection("mysql://different-server/other-db")

# Prove they're the same object
print(f"\ndb1 is db2: {db1 is db2}")  # True!
print(f"db1 id: {id(db1)}")
print(f"db2 id: {id(db2)}")
print(f"\nConnection string (same for both): {db1.connection_string}")

# Operations on db1 affect db2 (same object!)
db1.connect()
print(f"db2 connected: {db2.connected}")  # True!

---
## 6. Strategy Pattern - Swappable Algorithms

### What is Strategy Pattern?
- Define a **family of algorithms**
- Make them **interchangeable**
- Client code doesn't know which specific algorithm is used

In [ ]:
class HashingStrategy(ABC):
    """Abstract strategy - all hashing algorithms must implement hash()."""
    
    @abstractmethod
    def hash(self, value: str) -> str:
        pass


class MD5Strategy(HashingStrategy):
    """Concrete strategy: MD5 hashing (fast, not secure)."""
    
    def hash(self, value: str) -> str:
        import hashlib
        result = hashlib.md5(value.encode()).hexdigest()
        print(f"   [MD5] {value} → {result[:16]}...")
        return result


class SHA256Strategy(HashingStrategy):
    """Concrete strategy: SHA256 hashing (slower, more secure)."""
    
    def hash(self, value: str) -> str:
        import hashlib
        result = hashlib.sha256(value.encode()).hexdigest()
        print(f"   [SHA256] {value} → {result[:16]}...")
        return result


class HashingService:
    """
    CONTEXT: Uses a strategy but doesn't know which one.
    Strategy can be swapped at runtime!
    """
    
    def __init__(self, strategy: HashingStrategy):
        self._strategy = strategy
    
    def set_strategy(self, strategy: HashingStrategy):
        """Swap the algorithm at runtime!"""
        print(f"🔄 Switching to {strategy.__class__.__name__}")
        self._strategy = strategy
    
    def hash_value(self, value: str) -> str:
        """Delegates to whatever strategy is currently set."""
        return self._strategy.hash(value)

In [ ]:
# DEMO: Swap algorithms at runtime

# Start with fast MD5
service = HashingService(MD5Strategy())
print("Using MD5:")
service.hash_value("password123")

# Switch to secure SHA256
print("\nSwitching algorithm...")
service.set_strategy(SHA256Strategy())
print("Using SHA256:")
service.hash_value("password123")

# BENEFIT: Client code (service.hash_value) never changed!
# Only the internal algorithm changed.

---
## 7. Decorator Pattern - Adding Behavior

### What are Python Decorators?
- Functions that **wrap other functions**
- Add behavior **without modifying original code**
- Examples: logging, timing, retry logic, authentication

In [5]:
def timer(func):
    """
    DECORATOR: Measures how long a function takes.
    
    HOW IT WORKS:
    1. @timer is applied to a function
    2. Returns a WRAPPER function that:
       - Records start time
       - Calls original function
       - Records end time
       - Prints elapsed time
    """
    @functools.wraps(func)  # Preserves original function's name and docstring
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)  # Call original function
        elapsed = time.time() - start
        print(f"⏱️ {func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper


def logger(func):
    """
    DECORATOR: Logs function calls with arguments and return value.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📝 Calling {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)
        print(f"📝 {func.__name__} returned: {result}")
        return result
    return wrapper

In [4]:
# APPLYING DECORATORS

@timer          # This decorator runs SECOND (outer)
@logger         # This decorator runs FIRST (inner)
def slow_calculation(x: int, y: int) -> int:
    """Simulates a slow calculation."""
    time.sleep(0.1)  # Simulate work
    return x + y

# The above is equivalent to:
# slow_calculation = timer(logger(slow_calculation))

print("Calling decorated function:")
print("="*50)
result = slow_calculation(5, 3)
print(f"\nFinal result: {result}")

NameError: name 'functools' is not defined

In [ ]:
# DECORATOR WITH ARGUMENTS (decorator factory)

def retry(max_attempts: int = 3, delay: float = 0.1):
    """
    DECORATOR FACTORY: Returns a decorator with configurable parameters.
    
    Usage: @retry(max_attempts=5, delay=0.5)
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    print(f"   ⚠️ Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            
            print(f"   ❌ All {max_attempts} attempts failed!")
            raise last_exception
        return wrapper
    return decorator


@retry(max_attempts=3, delay=0.1)
def unreliable_api_call():
    """Simulates an unreliable API that sometimes fails."""
    import random
    if random.random() < 0.7:  # 70% chance of failure
        raise ConnectionError("API unavailable")
    return {"status": "success"}

print("Calling unreliable API with retry:")
try:
    result = unreliable_api_call()
    print(f"✅ Success: {result}")
except ConnectionError:
    print("❌ All retries exhausted")

---
## Summary: When to Use Each Pattern

| Pattern | Use When | Example |
|---------|----------|--------|
| Encapsulation | Protect internal state | Database credentials |
| Abstract Class | Define common interface | ETL processors |
| Dataclass | Store structured data | Claim, User, Config |
| Factory | Create objects dynamically | Processor for file type |
| Singleton | Only one instance needed | DB connection pool |
| Strategy | Swap algorithms | Hashing, compression |
| Decorator | Add cross-cutting concerns | Logging, timing, retry |